# I\. Setup and Album Title Inspection

## I\.1 Imports and DataFrame Upload

In [1]:
# Standard library imports
import os
from datetime import datetime

# Third-party imports
import numpy as np
import pandas as pd

os.chdir("/work")
print(os.listdir("./pipeline"))


['1.1.TMDB 2015-2025.csv', '2.1.MUSICBRAINZ_mv_tmdb_soundtrack_album_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_album_artist_bridge_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_artist_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_track_spine_2015_2025.csv', '2.2.MUSICBRAINZ_mv_tmdb_soundtrack_wide_track_2015_2025.csv', '3.2.Artists_official_df.csv', '3.2.Tracks_official_df.csv', '3.2.Wide_official_df.csv', '3.3.Albums_deduped_df.csv', '3.3.Artists_deduped_df.csv', '3.3.Tracks_deduped_df.csv', '3.3.Wide_deduped_df.csv', '3.4.Albums_canonical_identified_df.csv', '3.4.Artists_canonical_identified_df.csv', '3.4.Tracks_canonical_identified_df.csv', '3.4.Wide_canonical_identified_df.csv', '3.5.Albums_exploded_genre.csv', '3.5.Artists_exploded_genre.csv', '3.5.Tracks_exploded_genre.csv', '3.5.Wide_exploded_genre.csv', '3.6.Albums_vote_count_analysis.csv', '3.6.Artists_vote_count_analysis.csv', '3.6.Tracks_vote_count_analysis.csv', '3.6.Wide_vote_count_analysi

In [2]:
# Load the albums dataframe

album_df = pd.read_csv("./pipeline/3.3.Albums_deduped_df.csv")
display(album_df.shape)
display(album_df.columns)
display(album_df.head())

(4771, 63)

Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name', 'film_imdb_id',
       'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date',
       'film_popularity', 'film_budget', 'film_revenue', 'film_studios',
       'film_director', 'film_soundtrack_composer_raw', 'film_top_cast',
       'film_keywords', 'film_ingested_at', 'match_method', 'soundtrack_type',
       'notes', 'matched_at', 'release_group_id', 'release_group_mbid',
       'album_title', 'rg_primary_type', 'rg_secondary_types', 'release_id',
       'release_mbid', 'release_title', 'release_status', 'release_packaging',
       'barcode', 'release_language', 'release_script', 'release_comment',
       'album_us_release_date', 'album_us_release_year',
       'album_us_release_month_min_observed',
       'album_us_release_day_min_observed', 'us_date_has_missing_month',
       

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,label_mbids,label_tags_text,rg_rating,rg_rating_count,release_meta_date_added,release_meta_info_url,release_meta_amazon_asin,release_meta_cover_art_presence,release_status_clean,keep_row
0,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.875,12222,R,Fifty Shades of Grey,English,...,9e4d8965-4fa4-4ce9-a48a-6db90769aa05 | e0ecd90...,NaN,NaN,NaN,2015-01-09 19:01:19.035 -0500,https://www.amazon.com/gp/product/B00S17FT82,NaN,present,Official,True
1,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.875,12222,R,Fifty Shades of Grey,English,...,1391bdc7-a22c-48a4-a5fb-e7b8ef6ce143 | e0ecd90...,alejandra guzman | clean up | es 2003 | sólo e...,NaN,NaN,2015-02-16 09:22:51.239 -0500,https://www.amazon.com/gp/product/B00S9303ZM,B00S9303ZM,present,Official,True
2,150689,Cinderella,False,105,"Romance, Fantasy, Family, Drama",6.826,7294,PG,Cinderella,English,...,1c50034a-5c92-41a4-95ad-0acb7ea4686f,score | soundtrack | soundtrack label | classi...,NaN,NaN,2015-03-16 23:29:40.780 -0400,https://www.amazon.com/gp/product/B00THRE5TO,B00THRE5TO,present,Official,True
3,281957,The Revenant,False,157,"Western, Drama, Adventure",7.540,18928,R,The Revenant,English,...,41099e29-7301-4fa3-86a4-8d510157b275,burbank | california | clean up | score | soun...,NaN,NaN,2015-12-31 15:04:24.681 -0500,https://www.amazon.com/gp/product/B018THTR7M,B018THTR7M,present,Official,True
4,140607,Star Wars: The Force Awakens,False,136,"Adventure, Action, Science Fiction",7.300,20206,PG-13,Star Wars: The Force Awakens,English,...,1c50034a-5c92-41a4-95ad-0acb7ea4686f,score | soundtrack | soundtrack label | classi...,93.0,3.0,2015-09-04 14:24:34.396 -0400,https://www.amazon.com/gp/product/B014V6JIQK,B014V6JIQK,present,Official,True


## I\.2 Inspect album titles

Question: Are there identifying features in the album\_title that allow us to determine which albums are orchestral "soundtracks" \(the score of an album or a compilation of songs or something else?

In [3]:
TITLE_PATTERNS = {
    "has_inspired_by": r"inspired by",
    "has_score": r"orchestral score|score",
    "has_songs": r"soundtrack|music from|songs from",
    "has_from_motion_picture_phrase": r"from the motion picture|from the original motion picture",
    "has_edition_marker": r"deluxe|expanded|remastered|anniversary"
}

album_titles = (
    album_df[["release_group_mbid", 
    "tmdb_id", 
    "album_title"]]
    .copy()
)

for k, pattern in TITLE_PATTERNS.items():
    album_titles[k] = album_titles["album_title"].str.lower().str.contains(
        pattern, regex=True
    )

# How prevalent is each designation?
inspection_summary = album_titles[list(TITLE_PATTERNS.keys())].mean().round(3)
inspection_summary

has_inspired_by                   0.004
has_score                         0.051
has_songs                         0.402
has_from_motion_picture_phrase    0.025
has_edition_marker                0.000
dtype: float64

Findings: Album titles frequently include song\-based indicators \(e\.g\., “soundtrack,” “music from”\), appearing in roughly 40% of cases, while explicit references to scores are far less common \(~5%\)\. Other variants, such as “inspired by” releases or special editions, are rare\. This uneven use of title semantics confirms that album titles alone are insufficient for consistently identifying a single, canonical soundtrack per film, motivating the use of film\-level rules to resolve multiple albums and support a clean, one\-soundtrack\-per\-film analytic frame\.

# II\. Determining the Canonical Soundtrack

We will ceate an append table that flags the canonical soundtrack per film \(is\_canonical\_soundtrack\) and, when present, the canonical songs compilation \(is\_canonical\_songtrack\), keyed by \(tmdb\_id, release\_group\_mbid\)\.

## II\.1 Setup Base Tables

Before applying canonical selection rules, we construct a lightweight working table that isolates the fields needed for soundtrack classification\. This step derives simple title\-based signals \(e\.g\., “score,” “soundtrack,” excluding “inspired by”\), counts how many albums are linked to each film, and initializes flags that will later formalize which release\_group represents the canonical soundtrack or song compilation\.

In [4]:

# 1. Construct a base table for the cleanup
base_cols = ["tmdb_id", "release_group_mbid", "album_title", "match_method", "film_imdb_id"]
base_find_canon = album_df.loc[:, base_cols].copy()

# 2. Pull out the relevant title hints when present
t = base_find_canon['album_title'].str.lower()
base_find_canon['title_has_score'] = t.str.contains("score", regex = False) # includes "original score", "orchestral score", etc.
base_find_canon["title_has_soundtrack"] = (
    t.str.contains("soundtrack", regex=False)
    | t.str.contains("music from", regex=False)
    | t.str.contains("songs from", regex=False)
)

# Exclude “inspired by” from being considered canonical
base_find_canon["title_has_inspired_by"] = t.str.contains("inspired by", regex=False)

# 3. Count albums per film and add it to each record
film_album_ct = base_find_canon.groupby('tmdb_id')['release_group_mbid'].count().rename('album_ct').reset_index()
base_find_canon = base_find_canon.merge(film_album_ct, on = 'tmdb_id', how = 'left')

#4. Initialize final output flags
base_find_canon['is_canonical_soundtrack'] = False
base_find_canon['is_canonical_songtrack'] = False
base_find_canon['canonical_rule'] = ""
base_find_canon['canonical_songtrack_rule'] = ""

display(base_find_canon.head(10))

,tmdb_id,release_group_mbid,album_title,match_method,film_imdb_id,title_has_score,title_has_soundtrack,title_has_inspired_by,album_ct,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule,canonical_songtrack_rule
0,216015,85852629-ce4c-4bb8-8d4a-50299d56a06c,Fifty Shades of Grey: Original Motion Picture ...,imdb_exact,tt2322441,False,True,False,2,False,False,,
1,216015,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,Fifty Shades of Grey: Original Motion Picture ...,imdb_exact,tt2322441,True,False,False,2,False,False,,
2,150689,412fdd65-c3f8-40fe-a420-8b438e048915,Cinderella,imdb_exact,tt1661199,False,False,False,1,False,False,,
3,281957,5d5a40d3-8aab-41a8-8cfb-b97688a5c10c,The Revenant: Original Motion Picture Soundtrack,imdb_exact,tt1663202,False,True,False,1,False,False,,
4,140607,405fd3c5-0a45-456a-b853-6f734d3b57aa,Star Wars: The Force Awakens: Original Motion ...,imdb_exact,tt2488496,False,True,False,2,False,False,,
5,140607,4220161f-f90f-482b-9707-5c864fa75a8d,Jabba Flow (from “Star Wars: The Force Awakens”),imdb_exact,tt2488496,False,False,False,2,False,False,,
6,273481,771042e4-5d18-4f47-a2bf-51bc51b37c2c,Sicario: Original Motion Picture Soundtrack,imdb_exact,tt3397884,False,True,False,1,False,False,,
7,257344,cbb59154-37a0-4fa0-ad53-1f754040d03a,Pixels: The Movie (Original Motion Picture Sou...,imdb_exact,tt2120120,False,True,False,1,False,False,,
8,214756,faa0a8e5-03fa-43de-a347-99e164799ba4,Ted 2: Original Motion Picture Soundtrack,imdb_exact,tt2637276,False,True,False,1,False,False,,
9,238713,80071628-b7e2-4640-861e-b5338d0e2e41,Spy (Original Motion Picture Soundtrack),imdb_exact,tt3079380,False,True,False,2,False,False,,


Findings: This setup confirms that title\-based signals behave as expected: albums explicitly labeled “Original Motion Picture Soundtrack” are correctly flagged as soundtrack candidates, while “Original Motion Picture Score” is captured separately\. Films with multiple albums \(e\.g\., soundtrack \+ score, or soundtrack \+ single release\) are clearly identifiable via album\_ct, giving us the structure needed to programmatically select a single canonical soundtrack or song compilation in the next step\.

## II\.2 Helper Functions for Reporting

In [5]:

# Let's define a helper function to keep track of how many of our films are associated with exactly 1
# canonical soundtrack
def pct_films_with_exactly_one_canon(df):
    canon_ct = df.groupby("tmdb_id")["is_canonical_soundtrack"].sum()   # count of True per film
    return (canon_ct == 1).mean()

# Just in case we accidentally match more than 1 canonical soundtrack with the film.
# We are clear if the results of both functions match
def pct_films_with_any_canon(df):
    canon_any = df.groupby("tmdb_id")["is_canonical_soundtrack"].max()  # True if any row in film is True
    return canon_any.mean()


## II\.3 Rule 1: Single album films automatically become the canonical soundtrack

We start canon selection with the simplest, highest\-confidence rule: if a film only has one album candidate in our spine, there’s nothing to resolve — that album becomes the canonical soundtrack by definition\. This cell applies that rule and reports how much of the film set is immediately “done” before moving to trickier multi\-album cases\.

In [6]:

# --------------------------------------------
# Rule 1: single-album films => canonical soundtrack
# --------------------------------------------
# If a film has exactly 1 candidate album in the spine, that album is canonical by definition.

print(f"Before Rule 1 (any canon):     {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"Before Rule 1 (exactly one):   {pct_films_with_exactly_one_canon(base_find_canon):.1%}")

# Films with exactly one album in the candidate universe
mask_single = base_find_canon["album_ct"] == 1

base_find_canon.loc[mask_single, "is_canonical_soundtrack"] = True
base_find_canon.loc[mask_single, "canonical_rule"] = "single_album_film"

print(f"After Rule 1 (any canon):      {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"After Rule 1 (exactly one):    {pct_films_with_exactly_one_canon(base_find_canon):.1%}")
print(f"Rule 1 picks applied (films):  {base_find_canon.loc[mask_single, 'tmdb_id'].nunique()}")


Before Rule 1 (any canon):     0.0%
Before Rule 1 (exactly one):   0.0%
After Rule 1 (any canon):      93.7%
After Rule 1 (exactly one):    93.7%
Rule 1 picks applied (films):  4166


Findings: Rule 1 does most of the work immediately: 93\.7% of films have only a single candidate album, allowing us to canonize 4,166 films in one pass\. The remaining ~6% are the genuinely ambiguous multi\-album cases that require more nuanced rules\.

## II\.4 Rule 2: Multi\-album films with "soundtrack" and "score" albums

Next we tackle the multi\-album films by using simple title cues\. We flag likely score albums as canonical soundtrack candidates and likely soundtrack/music\-from/songs\-from albums as songtrack candidates—without forcing a one\-per\-film decision yet\.

In [7]:

# Rule 2 (multi-album title hints)
# -------------------------------
# For films with multiple albums:
# - If title indicates a "score", mark as canonical soundtrack candidate.
# - If title indicates a "soundtrack/music from/songs from", mark as songtrack candidate.
#
# NOTE: At this point, we are only flagging candidates. We have NOT enforced
# the "exactly one" constraint yet. That happens in resolve_per_film().

print(f"Before Rule 2 (any canon):     {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"Before Rule 2 (exactly one):   {pct_films_with_exactly_one_canon(base_find_canon):.1%}")

mask_multi = base_find_canon["album_ct"] > 1

# Score => canonical soundtrack
score_mask = mask_multi & base_find_canon["title_has_score"] & (~base_find_canon["title_has_inspired_by"])
base_find_canon.loc[score_mask, "is_canonical_soundtrack"] = True
base_find_canon.loc[score_mask, "canonical_rule"] = "multi_album_score_title"

# Soundtrack/music-from/songs-from => canonical songtrack
songs_mask = mask_multi & base_find_canon["title_has_soundtrack"] & (~base_find_canon["title_has_inspired_by"])
base_find_canon.loc[songs_mask, "is_canonical_songtrack"] = True
base_find_canon.loc[songs_mask, "canonical_songtrack_rule"] = "multi_album_soundtrack_title"

print(f"After Rule 2 (any canon):      {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"After Rule 2 (exactly one):    {pct_films_with_exactly_one_canon(base_find_canon):.1%}")
print(f"Rule 2 picks applied (albums): {int(score_mask.sum()):,} score flags | {int(songs_mask.sum()):,} songtrack flags")
print(f"Rule 2 picks applied (films):  {base_find_canon.loc[score_mask, 'tmdb_id'].nunique():,} score films | {base_find_canon.loc[songs_mask, 'tmdb_id'].nunique():,} songtrack films")


Before Rule 2 (any canon):     93.7%
Before Rule 2 (exactly one):   93.7%
After Rule 2 (any canon):      95.8%
After Rule 2 (exactly one):    95.8%
Rule 2 picks applied (albums): 96 score flags | 219 songtrack flags
Rule 2 picks applied (films):  95 score films | 174 songtrack films


Findings: Rule 2 adds a meaningful bump in coverage, raising canonized films from 93\.7% to 95\.8%\. It flags 96 score albums \(95 films\) and 219 songtrack albums \(174 films\), capturing many of the “soundtrack vs score” multi\-album cases\.

## II\.5 Rule 3: IMDB exact matches

For the remaining edge cases, we apply a pragmatic fallback\. If a film still lacks a canonical soundtrack after Rules 1–2, we select an album with an imdb\_exact match as the canonical soundtrack\. This leverages our highest\-confidence linkage signal to close remaining gaps without overfitting\.

In [8]:

# Rule 3 (fallback: IMDB exact match)
# ----------------------------------
# If a film still has 0 canonical soundtracks after Rules 1–2, and one of its albums is an
# IMDB exact match, mark that album as the canonical soundtrack.
#
# Note: a small number of films have >1 imdb_exact match; we keep the first deterministically.

print(f"Before Rule 3 (any canon):     {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"Before Rule 3 (exactly one):   {pct_films_with_exactly_one_canon(base_find_canon):.1%}")

# Films still missing a canonical soundtrack
films_needing_rule3 = base_find_canon.groupby("tmdb_id")["is_canonical_soundtrack"].sum()
films_needing_rule3 = films_needing_rule3[films_needing_rule3 == 0].index

# Pick one imdb_exact album per film (deterministic)
rule3_picks = (
    base_find_canon[
        base_find_canon["tmdb_id"].isin(films_needing_rule3) &
        (base_find_canon["match_method"] == "imdb_exact")
    ][["tmdb_id", "release_group_mbid"]]
    .sort_values(["tmdb_id", "release_group_mbid"])
    .drop_duplicates(subset=["tmdb_id"])
)

# Apply picks using (tmdb_id, release_group_mbid) key membership
picked = base_find_canon.set_index(["tmdb_id", "release_group_mbid"]).index.isin(
    rule3_picks.set_index(["tmdb_id", "release_group_mbid"]).index
)

base_find_canon.loc[picked, "is_canonical_soundtrack"] = True
base_find_canon.loc[picked, "canonical_rule"] = "fallback_imdb_match"

print(f"After Rule 3 (any canon):      {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"After Rule 3 (exactly one):    {pct_films_with_exactly_one_canon(base_find_canon):.1%}")
print(f"Rule 3 picks applied (films):  {len(rule3_picks)}")


Before Rule 3 (any canon):     95.8%
Before Rule 3 (exactly one):   95.8%
After Rule 3 (any canon):      96.7%
After Rule 3 (exactly one):    96.7%
Rule 3 picks applied (films):  42


Findings: Rule 3 provides a final incremental lift, increasing canon coverage from 95\.8% to 96\.7% by resolving 42 additional films\. At this stage, nearly the entire film set has exactly one canonical soundtrack identified, with only a small residual set remaining ambiguous\.

## II\.6 Rule 4: Everything else: first release group becomes canonical

Finally, for the small set of films still unresolved after Rules 1–3, we apply a deterministic fallback\. Rather than introducing additional heuristics, we select a single release\_group per film using a stable ordering rule \(alphabetically smallest release\_group\_mbid\) to guarantee complete and reproducible coverage\.

In [9]:

# Rule 4 (fallback: earliest release_group_mbid when still ambiguous)
# ---------------------------------------------------------------
# If a film still has 0 canonical soundtracks after Rule 3, pick a stable deterministic default:
# the earliest (alphabetically smallest) release_group_mbid within that tmdb_id.
#
# Coverage is reported the same way as Rule 3: film-level before/after using our helper functions.

print(f"Before Rule 4 (any canon):     {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"Before Rule 4 (exactly one):   {pct_films_with_exactly_one_canon(base_find_canon):.1%}")

# Films still missing a canonical soundtrack
films_still_unassigned = base_find_canon.groupby("tmdb_id")["is_canonical_soundtrack"].sum()
films_still_unassigned = films_still_unassigned[films_still_unassigned == 0].index

# Pick one album per film deterministically (earliest release_group_mbid)
rule4_picks = (
    base_find_canon[
        base_find_canon["tmdb_id"].isin(films_still_unassigned)
    ][["tmdb_id", "release_group_mbid"]]
    .sort_values(["tmdb_id", "release_group_mbid"])
    .drop_duplicates(subset=["tmdb_id"])
)

# Apply picks using (tmdb_id, release_group_mbid) key membership
picked = base_find_canon.set_index(["tmdb_id", "release_group_mbid"]).index.isin(
    rule4_picks.set_index(["tmdb_id", "release_group_mbid"]).index
)

base_find_canon.loc[picked, "is_canonical_soundtrack"] = True
base_find_canon.loc[picked, "canonical_rule"] = "fallback_earliest_mbid"

print(f"After Rule 4 (any canon):      {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"After Rule 4 (exactly one):    {pct_films_with_exactly_one_canon(base_find_canon):.1%}")
print(f"Rule 4 picks applied (films):  {len(rule4_picks)}")


Before Rule 4 (any canon):     96.7%
Before Rule 4 (exactly one):   96.7%
After Rule 4 (any canon):      100.0%
After Rule 4 (exactly one):    100.0%
Rule 4 picks applied (films):  145


Findings: Rule 4 closes the remaining gap, raising coverage from 96\.7% to 100%\. A total of 145 films required this deterministic fallback, ensuring that every film in the dataset now has exactly one canonical soundtrack assignment\.

As a final guardrail, we run a safety pass to enforce the one\-film, one\-canonical\-soundtrack constraint\. If any film somehow has multiple canonical flags after the rule cascade, we retain a single deterministic winner and unflag the rest\.

In [10]:
# ------------------------------------------------------------
# Safety pass (simple): if a film has >1 canon flagged, keep ONE
# ------------------------------------------------------------

print(f"Before safety pass (any canon):     {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"Before safety pass (exactly one):   {pct_films_with_exactly_one_canon(base_find_canon):.1%}")

# Only look at rows currently flagged as canonical soundtrack
canon_rows = base_find_canon[base_find_canon["is_canonical_soundtrack"]].copy()

# For each tmdb_id, keep the first canonical row; everything else gets unflagged
keep_idx = canon_rows.groupby("tmdb_id", sort=False).head(1).index
drop_idx = canon_rows.index.difference(keep_idx)

base_find_canon.loc[drop_idx, "is_canonical_soundtrack"] = False
base_find_canon.loc[drop_idx, "canonical_rule"] = "safety_pass_dropped"

# (Optional) tag the winners so we can tell later
base_find_canon.loc[keep_idx, "canonical_rule"] = (
    base_find_canon.loc[keep_idx, "canonical_rule"].fillna("safety_pass_winner")
)

print(f"Safety pass: dropped extra canon rows: {len(drop_idx)}")

print(f"After safety pass (any canon):      {pct_films_with_any_canon(base_find_canon):.1%}")
print(f"After safety pass (exactly one):    {pct_films_with_exactly_one_canon(base_find_canon):.1%}")

# Quick peek at what got dropped
display(
    base_find_canon.loc[drop_idx, ["tmdb_id", "release_group_mbid", "album_title", "match_method", "canonical_rule"]]
    .head(25)
)


Before safety pass (any canon):     100.0%
Before safety pass (exactly one):   100.0%
Safety pass: dropped extra canon rows: 1
After safety pass (any canon):      100.0%
After safety pass (exactly one):    100.0%


,tmdb_id,release_group_mbid,album_title,match_method,canonical_rule
4155,933260,cd19ba44-2bbf-480e-8c40-1432b3628713,The Substance: Original Motion Picture Score,title_contains_strict,safety_pass_dropped


Findings: The safety pass removed just one extra canonical row, confirming that the rule cascade was already well\-behaved\. Coverage remains at 100%, with exactly one canonical soundtrack per film\.

# III\. Create the append table and validate

## III1\. Append Table

With the canonical selection logic finalized, we materialize a clean film–album append table at the \(tmdb\_id, release\_group\_mbid\) grain\. This table carries the canonical flags and rule provenance, allowing us to verify key uniqueness before merging the results back onto album\_df and locking in a structurally sound, one\-canonical\-per\-film layer\.

In [11]:
# Append table + quick checks
# ---------------------------
# Now that we’ve applied Rules 1–4, we can materialize the append table at film–album grain.
# Primary key: (tmdb_id, release_group_mbid)
# Requirement: exactly 1 canonical soundtrack per film (tmdb_id)

soundtrack_append = base_find_canon.loc[:, [
    "tmdb_id",
    "release_group_mbid",
    "album_title",
    "album_ct",
    "is_canonical_soundtrack",
    "is_canonical_songtrack",
    "canonical_rule",
    "canonical_songtrack_rule"
]].copy()

display(soundtrack_append.head(25))

,tmdb_id,release_group_mbid,album_title,album_ct,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule,canonical_songtrack_rule
0,216015,85852629-ce4c-4bb8-8d4a-50299d56a06c,Fifty Shades of Grey: Original Motion Picture ...,2,False,True,,multi_album_soundtrack_title
1,216015,fa7aa3a0-3de5-4273-bbb4-cfc13dbf9f1e,Fifty Shades of Grey: Original Motion Picture ...,2,True,False,multi_album_score_title,
2,150689,412fdd65-c3f8-40fe-a420-8b438e048915,Cinderella,1,True,False,single_album_film,
3,281957,5d5a40d3-8aab-41a8-8cfb-b97688a5c10c,The Revenant: Original Motion Picture Soundtrack,1,True,False,single_album_film,
4,140607,405fd3c5-0a45-456a-b853-6f734d3b57aa,Star Wars: The Force Awakens: Original Motion ...,2,True,True,fallback_imdb_match,multi_album_soundtrack_title
5,140607,4220161f-f90f-482b-9707-5c864fa75a8d,Jabba Flow (from “Star Wars: The Force Awakens”),2,False,False,,
6,273481,771042e4-5d18-4f47-a2bf-51bc51b37c2c,Sicario: Original Motion Picture Soundtrack,1,True,False,single_album_film,
7,257344,cbb59154-37a0-4fa0-ad53-1f754040d03a,Pixels: The Movie (Original Motion Picture Sou...,1,True,False,single_album_film,
8,214756,faa0a8e5-03fa-43de-a347-99e164799ba4,Ted 2: Original Motion Picture Soundtrack,1,True,False,single_album_film,
9,238713,80071628-b7e2-4640-861e-b5338d0e2e41,Spy (Original Motion Picture Soundtrack),2,False,True,,multi_album_soundtrack_title


There really shouldn't be any dupes across the composite key of \(tmdb\_id, release\_group\_mbid\),
but just in case \-\- let's validate\.

In [12]:
album_key_dupes = album_df.duplicated(["tmdb_id", "release_group_mbid"]).sum()
print("album_df duplicate (tmdb_id, release_group_mbid) keys:", album_key_dupes)

append_key_dupes = soundtrack_append.duplicated(["tmdb_id", "release_group_mbid"]).sum()
print("soundtrack_append duplicate keys:", append_key_dupes)


album_df duplicate (tmdb_id, release_group_mbid) keys: 0
soundtrack_append duplicate keys: 0


Confirmed that there are no dupes indeed\! So we are ready to merge\.\.\.

In [13]:
# Merge canonical flags back onto album_df (no need to rebuild soundtrack_append if it already exists)
album_df_before = len(album_df)

album_mrg_df = album_df.merge(
    soundtrack_append[[
        "tmdb_id",
        "release_group_mbid",
        "is_canonical_soundtrack",
        "is_canonical_songtrack",
        "canonical_rule",
        "canonical_songtrack_rule"
    ]].drop_duplicates(["tmdb_id", "release_group_mbid"]),
    on=["tmdb_id", "release_group_mbid"],
    how="left",
    validate="1:1"   # safe even if album_df happens to be 1:1
)

print("album_df rows before:", album_df_before)
print("album_df rows after: ", len(album_mrg_df))


album_df rows before: 4771
album_df rows after:  4771


The merge preserves row integrity: album\_df remains at 4,771 rows before and after the canonical flag append, confirming a clean 1:1 merge at the \(tmdb\_id, release\_group\_mbid\) level with no duplication or unintended row expansion\.

## III\.2 Validation

In [14]:

# -----------------------------------------
# Compare canonical soundtrack count vs films
# -----------------------------------------

# Number of canonical soundtracks flagged (row-level)
canon_soundtrack_ct = album_mrg_df["is_canonical_soundtrack"].sum()

# Number of unique films
film_ct = album_mrg_df["tmdb_id"].nunique()

print(f"Canonical soundtracks flagged (rows): {int(canon_soundtrack_ct):,}")
print(f"Unique films (tmdb_id):                {film_ct:,}")
print(f"Difference (canon - films):            {int(canon_soundtrack_ct - film_ct):,}")


Canonical soundtracks flagged (rows): 4,448
Unique films (tmdb_id):                4,448
Difference (canon - films):            0


Hmm, there is still one straggler

In [15]:
# -----------------------------------------
# Find films with >1 canonical soundtrack
# -----------------------------------------

canon_per_film = album_mrg_df.groupby("tmdb_id")["is_canonical_soundtrack"].sum()

bad_tmdb_ids = canon_per_film[canon_per_film > 1].index.tolist()
print("Films with >1 canonical soundtrack:", len(bad_tmdb_ids))
print("Bad tmdb_ids:", bad_tmdb_ids[:20])

bad_rows = album_mrg_df.loc[
    album_mrg_df["tmdb_id"].isin(bad_tmdb_ids) & (album_mrg_df["is_canonical_soundtrack"]),
    ["tmdb_id", "film_title", "release_group_mbid", "album_title", "match_method", "canonical_rule"]
].sort_values(["tmdb_id", "canonical_rule", "release_group_mbid"])

display(bad_rows)


Films with >1 canonical soundtrack: 0
Bad tmdb_ids: []


,tmdb_id,film_title,release_group_mbid,album_title,match_method,canonical_rule


In [16]:
# PK uniqueness
print("PK dupes:", soundtrack_append.duplicated(["tmdb_id", "release_group_mbid"]).sum())

PK dupes: 0


In [17]:
base_find_canon["canonical_rule"].value_counts(dropna=False)

canonical_rule
single_album_film          4166
                            322
fallback_earliest_mbid      145
multi_album_score_title      95
fallback_imdb_match          42
safety_pass_dropped           1
Name: count, dtype: int64

In [18]:
canon_per_film = base_find_canon.groupby("tmdb_id")["is_canonical_soundtrack"].sum()
canon_per_film.value_counts().sort_index()


is_canonical_soundtrack
1    4448
Name: count, dtype: int64

In [19]:
# Visually inspect the fallback matches
display(base_find_canon[base_find_canon["canonical_rule"].str.contains("fallback", na=False)]
        .loc[:, ["tmdb_id","album_title","is_canonical_soundtrack","is_canonical_songtrack","canonical_rule"]
        ].sort_values('tmdb_id')
        )


,tmdb_id,album_title,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule
1406,38700,Bad Boys for Life: The Soundtrack,True,True,fallback_earliest_mbid
4072,47971,xXx: Return of Xander Cage,True,False,fallback_earliest_mbid
2527,136799,Trolls: Original Motion Picture Soundtrack,True,True,fallback_imdb_match
4,140607,Star Wars: The Force Awakens: Original Motion ...,True,True,fallback_imdb_match
3471,225728,Macbeth: Original Motion Picture Soundtrack,True,True,fallback_earliest_mbid
...,...,...,...,...,...
852,1230368,Springsteen:Deliver Me From Nowhere (Original ...,True,True,fallback_earliest_mbid
829,1236419,Piece By Piece,True,False,fallback_earliest_mbid
905,1275232,New (Original Motion Picture Soundtrack),True,True,fallback_earliest_mbid
1853,1293244,Four Letters of Love: Original Film Soundtrack,True,True,fallback_earliest_mbid


A quick spot check confirms that canonical selections are largely intuitive and rule\-driven, with most resolved via fallback\_imdb\_match or the deterministic fallback\. A few albums are flagged as both canonical soundtrack and songtrack \(e\.g\., traditional OSTs\), which is expected for releases that function as both score and compilation\. Overall, the outputs look structurally consistent with our rule hierarchy rather than arbitrary assignments\.

## III\.3 Songtrack exploration \(not as important\)

In [20]:
song_per_film = base_find_canon.groupby("tmdb_id")["is_canonical_songtrack"].sum()
song_per_film.value_counts().sort_index()


is_canonical_songtrack
0    4274
1     137
2      31
3       4
4       2
Name: count, dtype: int64

In [21]:
# Songtrack sanity check (Might be more than 1 per film)
song_check = soundtrack_append.groupby("tmdb_id")["is_canonical_songtrack"].sum()
song_bad = song_check[song_check > 1]
print("Films with songtrack > 1:", len(song_bad))
display(song_bad.head(20))


Films with songtrack > 1: 37


tmdb_id
309242    2
339846    2
340103    2
347201    2
369523    2
377691    3
379686    2
402431    2
407447    2
419704    3
420817    3
425001    2
438631    2
458335    2
458576    2
493529    2
496186    2
504949    2
507089    4
513268    2
Name: is_canonical_songtrack, dtype: int64

# IV\. Two more extra calculations: films and album age

The film release dates and album release dates have been useful for matching film and soundtrack, but they are also useful as a potential feature for statistical analysis\. However, in their current form, probably not so much\. They would be more useful as age columns\. Let's compute the days\_since\_film\_release and days\_since\_album\_release columns

### IV\.1 Quick inspection

In [22]:
# Unfortunately, some of the release dates may be missing elements. Let's
# create a function to inspect missingness around a specific column

def inspect_date_missingness(df:pd.DataFrame, col:str):
    # 1) Raw missing (NaN / None / NaT)
    raw_missing_ct = df[col].isna().sum()
    raw_missing_pct = raw_missing_ct / len(album_mrg_df)

    # 2) Missing after parsing (catches blanks, bad strings, invalid dates)
    parsed = pd.to_datetime(df[col], errors="coerce", utc=True)
    parsed_missing_ct = parsed.isna().sum()
    parsed_missing_pct = parsed_missing_ct / len(album_mrg_df)

    # 3) (Optional) Treat empty/whitespace strings as missing explicitly (for visibility)
    empty_str_mask = df[col].astype("string").str.strip().eq("")
    empty_str_ct = empty_str_mask.sum()
    empty_str_pct = empty_str_ct / len(album_mrg_df)

    print(f"Total rows: {len(album_mrg_df):,}")
    print(f"Raw missing ({col} isna): {raw_missing_ct:,} ({raw_missing_pct:.2%})")
    print(f"Empty/blank strings:       {empty_str_ct:,} ({empty_str_pct:.2%})")
    print(f"Missing after to_datetime: {parsed_missing_ct:,} ({parsed_missing_pct:.2%})")

    # 4) Show a few problematic examples (non-missing raw, but unparseable)
    bad_examples = df.loc[~album_mrg_df[col].isna() & parsed.isna(), col].value_counts(dropna=False).head(20)
    print("\nTop unparseable raw values (up to 20):")
    display(bad_examples)

inspect_date_missingness(album_mrg_df, "film_release_date")


Total rows: 4,771
Raw missing (film_release_date isna): 0 (0.00%)
Empty/blank strings:       0 (0.00%)
Missing after to_datetime: 0 (0.00%)

Top unparseable raw values (up to 20):


Series([], Name: count, dtype: int64)

Findings: film\_release\_date is fully populated and clean\. There are no raw nulls, blank strings, or parsing failures, indicating 100% completeness and no malformed date values in the merged album layer\.

In [23]:
# Some dates may be incomplete. Let's inspect whether a date column has 
# year, month and day

def inspect_date_completeness(df:pd.DataFrame, col):
    raw = df[col].astype("string")
    s = raw.str.strip()

    parsed = pd.to_datetime(s, errors="coerce", utc=True)

    # Start with default label
    gran = pd.Series("unparseable", index=album_mrg_df.index, dtype="string")

    # Missing / blank
    gran.loc[s.isna() | s.eq("")] = "missing_blank"

    # Strict formats
    gran.loc[s.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)] = "yyyy-mm-dd"
    gran.loc[s.str.fullmatch(r"\d{4}-\d{2}", na=False)]       = "yyyy-mm"
    gran.loc[s.str.fullmatch(r"\d{4}", na=False)]             = "yyyy"

    # Parseable but not matching strict formats (e.g., "Mar 3 2020")
    gran.loc[parsed.notna() & ~s.isna() & ~s.eq("") &
            ~s.str.fullmatch(r"\d{4}(-\d{2}(-\d{2})?)?", na=False)] = "other_parseable"

    # Summarize
    summary = gran.value_counts(dropna=False).to_frame("count")
    summary["pct"] = summary["count"] / len(df)

    display(summary)

inspect_date_completeness(album_mrg_df, col = "film_release_date")

,count,pct
yyyy-mm-dd,4771,1.0


Findings: Film release date is fully populated, and every film listed has complete information in the format yyyy\-mm\-dd\. Converting it to 'days\_since\_film\_release' will be clean\.

Let's now inspect 'album\_us\_release\_date'

In [24]:
inspect_date_missingness(album_mrg_df, "album_us_release_date")

Total rows: 4,771
Raw missing (album_us_release_date isna): 3,347 (70.15%)
Empty/blank strings:       0 (0.00%)
Missing after to_datetime: 3,347 (70.15%)

Top unparseable raw values (up to 20):


Series([], Name: count, dtype: int64)

In [25]:
inspect_date_completeness(album_mrg_df, "album_us_release_date")

,count,pct
missing_blank,3347,0.70153
yyyy-mm-dd,1424,0.29847


Findings: It appears that 'album\_us\_release\_date' is missing from a lot of records \-\- only ~30% of records have them at this stage in the pipeline\. The good news is that 100% of the 1424 remaining records all have fully populated 'yyyy\-mm\-dd' values\. We might still be able to use a 'days\_since\_album\_age' attribute as a feature in our analysis, but it won't be that reliable\.

In [26]:
asof = pd.Timestamp.now(tz="UTC")   # Safeguard if notebook gets run in different timezones
album_mrg_df['days_since_film_release'] = (asof - pd.to_datetime(album_mrg_df['film_release_date'], errors='coerce', utc = True)).dt.days
display(album_mrg_df[['tmdb_id', 'film_title', 'film_release_date', 'days_since_film_release']].sample(10))

,tmdb_id,film_title,film_release_date,days_since_film_release
546,637649,Wrath of Man,2021-04-22,1758
4228,426789,A Doggone Christmas,2016-11-19,3373
3863,462469,Been So Long,2018-10-12,2681
907,490003,Won't You Be My Neighbor?,2018-06-29,2786
1670,381374,Action Hero Biju,2016-02-04,3662
3579,432618,Coexister,2017-10-11,3047
1944,318898,Alone,2015-01-16,4046
1281,793032,Cobra,2022-08-31,1262
2152,441862,Hebbuli,2017-02-23,3277
1601,412188,93 Days,2016-04-24,3582


In [27]:
album_mrg_df['days_since_album_release'] = (asof - pd.to_datetime(album_mrg_df['album_us_release_date'], errors='coerce', utc = True)).dt.days
display(album_mrg_df[['release_group_id', 'album_title','album_us_release_date','days_since_album_release']].sample(10))

,release_group_id,album_title,album_us_release_date,days_since_album_release
4585,1708283,The Finest Hours (Original Motion Picture Soun...,2016-01-29,3668.0
484,2194447,El sendero de la anaconda,NaN,NaN
2285,2289537,Queen & Slim: Original Motion Picture Score,NaN,NaN
2054,2507003,Military Wives (Original Motion Picture Soundt...,NaN,NaN
1140,2955672,Valimai,NaN,NaN
2375,2268087,Paradise War: The Story of Bruno Manser,NaN,NaN
3970,3564253,Joan Baez: I Am A Noise,NaN,NaN
4078,2579296,Chaman Bahaar,NaN,NaN
48,1483438,Roy,NaN,NaN
2468,1789824,Remember,2016-01-15,3682.0


# V\. Augment and export base tables

### V\.1 Setup Artist, Tracks and Wide

In [28]:
import shutil

shutil.copy(
    "./pipeline/3.3.Artists_deduped_df.csv",
    "./pipeline/3.4.Artists_canonical_identified_df.csv"
)

shutil.copy(
    "./pipeline/3.3.Tracks_deduped_df.csv",
    "./pipeline/3.4.Tracks_canonical_identified_df.csv"
)



'./pipeline/3.4.Tracks_canonical_identified_df.csv'

In [29]:
display(album_df.head())

album_mrg_df.to_csv(
     "./pipeline/3.4.Albums_canonical_identified_df.csv",
     index=False
)


,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,label_mbids,label_tags_text,rg_rating,rg_rating_count,release_meta_date_added,release_meta_info_url,release_meta_amazon_asin,release_meta_cover_art_presence,release_status_clean,keep_row
0,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.875,12222,R,Fifty Shades of Grey,English,...,9e4d8965-4fa4-4ce9-a48a-6db90769aa05 | e0ecd90...,NaN,NaN,NaN,2015-01-09 19:01:19.035 -0500,https://www.amazon.com/gp/product/B00S17FT82,NaN,present,Official,True
1,216015,Fifty Shades of Grey,False,125,"Drama, Romance, Thriller",5.875,12222,R,Fifty Shades of Grey,English,...,1391bdc7-a22c-48a4-a5fb-e7b8ef6ce143 | e0ecd90...,alejandra guzman | clean up | es 2003 | sólo e...,NaN,NaN,2015-02-16 09:22:51.239 -0500,https://www.amazon.com/gp/product/B00S9303ZM,B00S9303ZM,present,Official,True
2,150689,Cinderella,False,105,"Romance, Fantasy, Family, Drama",6.826,7294,PG,Cinderella,English,...,1c50034a-5c92-41a4-95ad-0acb7ea4686f,score | soundtrack | soundtrack label | classi...,NaN,NaN,2015-03-16 23:29:40.780 -0400,https://www.amazon.com/gp/product/B00THRE5TO,B00THRE5TO,present,Official,True
3,281957,The Revenant,False,157,"Western, Drama, Adventure",7.540,18928,R,The Revenant,English,...,41099e29-7301-4fa3-86a4-8d510157b275,burbank | california | clean up | score | soun...,NaN,NaN,2015-12-31 15:04:24.681 -0500,https://www.amazon.com/gp/product/B018THTR7M,B018THTR7M,present,Official,True
4,140607,Star Wars: The Force Awakens,False,136,"Adventure, Action, Science Fiction",7.300,20206,PG-13,Star Wars: The Force Awakens,English,...,1c50034a-5c92-41a4-95ad-0acb7ea4686f,score | soundtrack | soundtrack label | classi...,93.0,3.0,2015-09-04 14:24:34.396 -0400,https://www.amazon.com/gp/product/B014V6JIQK,B014V6JIQK,present,Official,True


### V\. 2 Wide Table augmentation with canonical flag

We really need to augment the wide table as well

In [30]:
# Load the wide dataframe

wide_df = pd.read_csv("./pipeline/3.3.Wide_deduped_df.csv")
display(wide_df.shape)
display(wide_df.columns)
display(wide_df.head())

/tmp/ipykernel_370/3112204062.py:3: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  wide_df = pd.read_csv("./pipeline/3.3.Wide_deduped_df.csv")


(78992, 88)

Index(['tmdb_id', 'film_title', 'film_adult', 'film_runtime_min',
       'film_genres', 'film_rating', 'film_vote_count', 'film_mpaa_rating',
       'film_original_title', 'film_language_name', 'film_imdb_id',
       'film_wikidata_id', 'film_countries', 'film_year', 'film_release_date',
       'film_popularity', 'film_budget', 'film_revenue', 'film_studios',
       'film_director', 'film_soundtrack_composer_raw', 'film_top_cast',
       'film_keywords', 'film_ingested_at', 'release_group_id',
       'release_group_mbid', 'release_id', 'release_mbid', 'album_title',
       'rg_primary_type', 'rg_secondary_types', 'match_method',
       'album_soundtrack_type', 'album_notes', 'album_matched_at',
       'album_us_release_date', 'album_us_release_year',
       'album_us_release_month_min_observed',
       'album_us_release_day_min_observed', 'us_date_has_missing_month',
       'us_date_has_missing_day', 'us_any_event_missing_month',
       'us_any_event_missing_day', 'release_title', 'rel

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,recording_artist_credit,isrcs_text,recording_first_release_year,recording_first_release_month,recording_first_release_day,recording_tags_text,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,Philippe Vachey,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,Philippe Vachey,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,Buc Fifty,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,Rob the Viking,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,Son Doobie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
wide_before = len(wide_df)

wide_mrg_df = wide_df.merge(
    soundtrack_append[[
        "tmdb_id",
        "release_group_mbid",
        "is_canonical_soundtrack",
        "is_canonical_songtrack",
        "canonical_rule",
        "canonical_songtrack_rule"
    ]].drop_duplicates(["tmdb_id", "release_group_mbid"]),
    on=["tmdb_id", "release_group_mbid"],
    how="left",
    validate="m:1"   # many track rows per (tmdb_id, release_group_mbid)
)

print("wide_df rows before:", wide_before)
print("wide_df rows after: ", len(wide_mrg_df))


wide_df rows before: 78992
wide_df rows after:  78992


In [32]:
# 1) Coverage: how many rows got canon flags populated at all?
print("Rows with is_canonical_soundtrack not-null:",
      wide_mrg_df["is_canonical_soundtrack"].notna().sum())

# 2) Sanity: canonical soundtrack rows should exist (unless you haven't flagged yet)
print("Rows flagged canonical soundtrack (track rows):",
      (wide_mrg_df["is_canonical_soundtrack"] == True).sum())


Rows with is_canonical_soundtrack not-null: 78992
Rows flagged canonical soundtrack (track rows): 74151


Finding: It looks a little shocking at first that most rows in the wide table are marked as coming from a canonical soundtrack\. In practice, this is mostly a side effect of the data structure\. Over 93% of films in the dataset only map to a single soundtrack album, so those albums are unambiguous and are marked canonical by definition\. When those albums are exploded out to the track level, each one contributes many rows, which makes the canonical flag look dominant in the wide table\. The skew is coming from track\-level expansion, not from aggressive or subjective classification\.

In [33]:
# Album-grain check inside wide_df:
#    Each film should map to exactly 1 canonical soundtrack album after resolution.
canon_album_ct_by_film = (
    wide_mrg_df.loc[wide_mrg_df["is_canonical_soundtrack"] == True]
      .groupby("tmdb_id")["release_group_mbid"]
      .nunique()
)

print("\nFilms w/ >=1 canonical soundtrack album (in wide_df):", (canon_album_ct_by_film >= 1).sum())
print("Films w/ exactly 1 canonical soundtrack album:", (canon_album_ct_by_film == 1).sum())
print("Films w/ >1 canonical soundtrack album:", (canon_album_ct_by_film > 1).sum())





Films w/ >=1 canonical soundtrack album (in wide_df): 4437
Films w/ exactly 1 canonical soundtrack album: 4437
Films w/ >1 canonical soundtrack album: 0


Finding: All films that appear in the wide table resolve cleanly to exactly one canonical soundtrack album\. This confirms that the canonicalization logic held up when expanded to the track level, with no remaining multi\-album ambiguity after the final safety pass\.

### V\.3 Calculate film and album ages for wide

In [34]:
before = len(wide_mrg_df.columns)
wide_mrg_df['days_since_film_release'] = (asof - pd.to_datetime(wide_mrg_df['film_release_date'], errors='coerce', utc = True)).dt.days
wide_mrg_df['days_since_album_release'] = (asof - pd.to_datetime(wide_mrg_df['album_us_release_date'], errors='coerce', utc = True)).dt.days
after = len(wide_mrg_df.columns)
print(f"Before wide_df column count: {before} and after: {after}")


Before wide_df column count: 92 and after: 94


In [35]:
# Quick inspection

display(wide_mrg_df[['tmdb_id', 'film_title', 'film_release_date', 'days_since_film_release']].sample(10))
display(wide_mrg_df[['release_group_id', 'album_title','album_us_release_date','days_since_album_release']].sample(10))

,tmdb_id,film_title,film_release_date,days_since_film_release
4865,258480,Carol,2015-11-20,3738
39645,734624,Emily and the Magical Journey,2020-09-17,1975
36994,614560,Mank,2020-11-13,1918
72705,575265,Mission: Impossible - The Final Reckoning,2025-05-17,272
22109,404368,Ralph Breaks the Internet,2018-11-20,2642
17835,456154,White Fang,2018-03-28,2879
17524,376540,Mathilde,2017-10-25,3033
11574,334531,My All American,2015-11-13,3745
75095,1205515,"Sorry, Baby",2025-06-27,231
77667,1241983,Train Dreams,2025-11-05,100


,release_group_id,album_title,album_us_release_date,days_since_album_release
72719,4191086,Out of My Mind: Original Soundtrack,NaN,NaN
29038,2263563,Frozen II,2019-11-15,2282.0
69258,4028540,The Last Kumite,2024-01-01,774.0
2644,1569144,Everest,2015-09-18,3801.0
22700,2075909,Vox Lux,NaN,NaN
57658,3404718,Boarding School,NaN,NaN
53122,3172589,The Fabelmans: Original Motion Picture Soundtrack,2022-12-27,1144.0
34673,2457303,The Outpost,2020-07-28,2026.0
49294,3046951,Freaks Out,NaN,NaN
14867,1883365,Justice League (Original Motion Picture Soundt...,NaN,NaN


### V\. 4 Export wide table

In [36]:
display(wide_mrg_df.head())

wide_mrg_df.to_csv(
     "./pipeline/3.4.Wide_canonical_identified_df.csv",
     index=False
)

,tmdb_id,film_title,film_adult,film_runtime_min,film_genres,film_rating,film_vote_count,film_mpaa_rating,film_original_title,film_language_name,...,work_ids_text,work_titles_text,composer_names_text,lyricist_names_text,is_canonical_soundtrack,is_canonical_songtrack,canonical_rule,canonical_songtrack_rule,days_since_film_release,days_since_album_release
0,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,,1225,NaN
1,1027160,Alone in the Dark,False,90,"Horror, Thriller",4.769,13,NR,Alone in the Dark,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,,1225,NaN
2,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,,2918,NaN
3,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,,2918,NaN
4,322855,Shakedown,False,72,Documentary,5.038,14,NR,Shakedown,English,...,NaN,NaN,NaN,NaN,True,False,single_album_film,,2918,NaN


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b124131d-2024-4253-bb46-8043aed4b78f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>